설치

In [ ]:
!pip install llama-index-core llama-index-llms-openai llama-index-embeddings-openai

API 키 설정

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("✅ API 키 설정 완료")

json 데이터 파일 로드 + document 변환 + 인덱싱

In [ ]:
import json
from llama_index.core import VectorStoreIndex, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.schema import Document

Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0.0)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

json_file_path = "drug_documents.json"

with open(json_file_path, "r", encoding="utf-8") as f:
    json_data = json.load(f)

documents = []

for item in json_data:
    page_content = item.get("page_content", "")
    metadata = item.get("metadata", {})

    documents.append(
        Document(
            text=page_content,
            metadata=metadata
        )
    )

print(f"✅ 불러온 문서 수: {len(documents)}")

# 인덱싱
print("📄 문서 인덱싱 중...")
index = VectorStoreIndex.from_documents(documents)
print("✅ 인덱싱 완료")

프롬프트 커스터마이징 + 쿼리 엔진 설정

In [ ]:
from llama_index.core.prompts import PromptTemplate

custom_prompt = PromptTemplate(
"""당신은 친근한 고양이 약사 챗봇입니다. 🐱💊
아래 참고 문서를 바탕으로만 답변하세요.

규칙:
- 반드시 문서에 있는 약만 사용
- 없는 약 절대 만들지 말 것
- 무조건 약을 추천하거나 판단할 것 (답변 안 하기 금지)
- 이해하기 쉽게 설명할 것

질문 유형:

1. [약 확인형]
(특정 약 이름 있음)

출력:
[약 이름]
-

[적합도]
- 적합 / 부적합 / 판단 어려움

[이유]
-

[주의사항]
-

[추가 안내]
-

---

2. [추천형]
(증상만 있음)

출력:
[추천 약]
-

[효능·효과]
-

[용법·용량]
-

[주의사항]
-

[부작용]
-

[성분]
-

[제형(외형)]
-

[추가 안내]
-

---------------------
참고 문서:
{context_str}
---------------------

질문:
{query_str}

답변:
"""
)

query_engine = index.as_query_engine(
    similarity_top_k=5,
    text_qa_template=custom_prompt,
    verbose=True
)

print("✅ 쿼리 엔진 준비 완료")

셀5 시나리오 테스트(일반질문)

In [ ]:
# ==============================
# 🧪 일반 질문 테스트
# ==============================

questions = [
    "머리가 너무 아픈데 어떤 약이 좋을까?",
    "타이레놀 한 번에 몇 알 먹어야 해?",
    "밥 안 먹었는데 타이레놀 먹어도 괜찮아?",
    "어린이용 해열제 추천해줘",
    "기침이 나는데 타이레놀콜드-에스정이 맞아?",
    "눈이 피로한데 어떤 약이 좋을까?",
    "속이 더부룩한데 먹을 수 있는 약 있어?",
    "타이레놀 말고 다른 두통약도 있어?"
]

print("=" * 70)
print("🧪 일반 질문 테스트")
print("=" * 70)

for i, question in enumerate(questions, 1):
    print("\n" + "-" * 70)
    print(f"질문 {i}. {question}")
    print("-" * 70)

    response = query_engine.query(question)

    print(response.response)

출력 결과

======================================================================
🧪 일반 질문 테스트
======================================================================

----------------------------------------------------------------------
질문 1. 머리가 너무 아픈데 어떤 약이 좋을까?
----------------------------------------------------------------------
[추천 약]
- 콜대원키즈이부펜시럽(이부프로펜)

[효능·효과]
- 머리가 아플 때 통증을 완화해주는 효과가 있습니다.

[용법·용량]
- 성인은 1회 1~2정을 드시고, 1일 3회까지 복용 가능합니다.

[주의사항]
- 복용 중 간기능이상, 혈액장애, 소화관 장애, 알레르기 반응 등의 증상이 나타날 경우 즉시 의사와 상의하세요.

[부작용]
- 복용 중 두통, 어지러움, 소화불량, 피부발진 등의 부작용이 나타날 수 있습니다.

[성분]
- 이부프로펜

[제형(외형)]
- 시럽

[추가 안내]
- 복용 전에 반드시 의사나 약사와 상의하시고, 정확한 용법과 용량을 지켜주세요. 복용 중 이상 증상이 나타날 경우 즉시 의사와 상의하시기 바랍니다.

----------------------------------------------------------------------
질문 2. 타이레놀 한 번에 몇 알 먹어야 해?
----------------------------------------------------------------------
[약 이름]
- 타이레놀콜드-에스정

[적합도]
- 판단 어려움

[이유]
- 복용량은 개인의 연령, 체중, 증상 등에 따라 다를 수 있습니다. 정확한 복용량은 의사 또는 약사와 상의하여야 합니다.

[주의사항]
- 복용량은 의사 또는 약사의 지시에 따라야 합니다.

[추가 안내]
- 복용 전에 의사 또는 약사와 상의하여 적절한 복용량을 확인하세요.

----------------------------------------------------------------------
질문 3. 밥 안 먹었는데 타이레놀 먹어도 괜찮아?
----------------------------------------------------------------------
[약 이름]
- 타이레놀정500밀리그람(아세트아미노펜)

[적합도]
- 판단 어려움

[이유]
- 타이레놀정500밀리그람(아세트아미노펜)은 감기로 인한 발열 및 동통(통증)에 사용되는 약으로, 밥을 먹지 않은 상태에서 복용해도 효과가 있을 수 있습니다. 그러나 개인의 건강 상태에 따라 다를 수 있으므로 의사 또는 약사와 상의하는 것이 좋습니다.

[주의사항]
- 매일 세 잔 이상 정기적 음주자가 이 약을 복용할 때는 의사 또는 약사와 상의해야 합니다. 만약 특이한 증상이 나타난다면 즉시 의사 또는 약사와 상의해야 합니다. 아세트아미노펜으로 일일 최대 용량(4,000 mg)을 초과하여 복용하지 말아야 합니다.

[추가 안내]
- 가능한 최단기간동안 최소 유효용량으로 복용하고, 1일 최대 8정(4 g)을 초과하여 복용하지 않도록 주의해야 합니다.식사와 함께 복용하면 위장 부작용을 줄일 수 있습니다.생각보다 많은 양을 복용했거나 이상 증상이 나타날 경우 즉시 의사 또는 약사와 상의해야 합니다.

----------------------------------------------------------------------
질문 4. 어린이용 해열제 추천해줘
----------------------------------------------------------------------
[추천 약]
- 어린이부루펜시럽(이부프로펜)

[효능·효과]
- 류마티양 관절염, 연소성 류마티양 관절염, 골관절염, 감기로 인한 발열 및 동통, 요통, 월경곤란증, 수술후 동통과 강직성 척추염, 두통, 치통, 근육통, 신경통, 급성통풍, 건선성 관절염, 연조직손상(염좌, 좌상), 비관절 류마티스질환(건염, 건초염, 활액낭염)에 사용

[용법·용량]
- 어린이의 경우 11~14세는 200-250 mg(10~13 ml)씩, 7~10세는 150~200 mg(8~10 ml)씩, 3~6세는 100~150 mg(5~8 ml), 1~2세는 50~100 mg(3~5 ml)씩, 1일 3~4회 복용

[주의사항]
- 체중이 30 kg미만인 어린이는 1일량이 500 mg(25 ml)을 초과해서는 안되며 공복에 복용을 피하는 것이 바람직

[부작용]
-

[성분]
- 이부프로펜

[제형(외형)]
- 시럽

[추가 안내]
- 의사 또는 약사와 상의 후 적절한 용법으로 복용해주세요.

----------------------------------------------------------------------
질문 5. 기침이 나는데 타이레놀콜드-에스정이 맞아?
----------------------------------------------------------------------
[약 이름]
- 타이레놀콜드-에스정

[적합도]
- 판단 어려움

[이유]
- 타이레놀콜드-에스정은 감기의 제증상 완화에 사용되는 약으로, 기침 증상에 대한 명시적인 효과가 없습니다. 기침이 주 증상이라면 다른 기침약을 고려해보는 것이 좋습니다.

[주의사항]
- 5~6회 복용하여도 증상이 좋아지지 않을 경우 의사 또는 약사와 상의하십시오.
- 정해진 용법과 용량을 잘 지키십시오.
- 어린이에게 투여할 경우 보호자의 지도·감독하에 투여하십시오.
- 복용 후 졸음이 나타날 수 있으므로 운전 및 기계조작 시 주의하십시오.
- 장기간 계속하여 복용하지 마십시오.

[추가 안내]
- 기침 증상에 대한 적합한 치료를 위해 의사 또는 약사와 상의하시는 것이 좋습니다.

----------------------------------------------------------------------
질문 6. 눈이 피로한데 어떤 약이 좋을까?
----------------------------------------------------------------------
[추천 약]
- 아이미루40이엑스점안액

[효능·효과]
- 눈의 피로, 결막충혈, 눈병예방, 자외선 및 기타 광선에 의한 눈의 염증 등에 사용

[용법·용량]
- 1일 3~6회, 1회 1~3적 점안

[주의사항]
- 30개월 이하의 유아는 사용하지 말 것
- 눈의 통증이 심한 환자, 안약에 알레르기 경험자, 녹내장 환자는 의사 또는 약사와 상의
- 소프트콘택트렌즈 착용 시 사용하지 말 것

[부작용]
- 눈의 충혈, 가려움, 부종, 피부의 발진 등이 나타나면 즉시 중지 후 의사와 상의

[성분]
- 레티놀팔미테이트, 피리독신염산염, 클로르페니라민말레산염, 네오스티그민메틸황산염, 테트라히드로졸린염산염, D-α-토코페롤아세테이트, L-아스파르트산칼륨

[제형(외형)]
- 점안액

[추가 안내]
- 정해진 용법과 용량을 잘 지키고 부작용이 나타날 경우 즉시 의사와 상의하세요.

----------------------------------------------------------------------
질문 7. 속이 더부룩한데 먹을 수 있는 약 있어?
----------------------------------------------------------------------
[추천 약]
- 노루모에프내복액

[효능·효과]
- 위산과다, 속쓰림, 위부불쾌감, 위부팽만감, 체함, 구역, 구토, 위통, 신트림, 식욕감퇴(식욕부진), 소화불량, 과식에 사용

[용법·용량]
- 만 15세 이상은 1회 1병 1일 3회, 만 11세 이상~만 15세 미만은 1회 2/3병 1일 3회, 만 8세 이상~만 11세 미만은 1회 1/2병 1일 3회, 만 5세 이상~만 8세 미만은 1회 1/3병 1일 3회, 만 3세 이상~만 5세 미만은 1회 1/4병 1일 3회 식후 복용

[주의사항]
- 만 3개월 미만의 젖먹이는 복용하지 마십시오. 만 1세 미만의 젖먹이, 카라멜에 과민증, 신장장애 환자, 나트륨 제한 식이를 하는 사람은 의사 또는 약사와 상의. 용법과 용량을 잘 지키십시오. 어린이에게 투여할 경우 보호자의 지도 감독하에 투여. 2주 정도 투여하여도 증상의 개선이 없을 경우 의사 또는 약사와 상의

[부작용]
- 변비 또는 설사 등이 나타나는 경우 의사 또는 약사와 상의

[성분]
- 탄산수소나트륨, 글리신, L-멘톨, 고추틴크

[제형(외형)]
- 액제

[추가 안내]
- 속이 더부룩할 때 위산과다, 속쓰림, 소화불량 등의 증상을 완화하기 위해 노루모에프내복액을 복용하시면 도움이 될 수 있습니다. 하지만 복용 전에 주의사항을 잘 숙지하고 의사 또는 약사와 상의하시는 것이 좋습니다.

----------------------------------------------------------------------
질문 8. 타이레놀 말고 다른 두통약도 있어?
----------------------------------------------------------------------
[약 이름]
- 테라플루데이타임건조시럽

[적합도]
- 적합

[이유]
- 감기의 여러 증상(콧물, 코막힘, 재채기, 인후통, 오한, 발열, 두통, 관절통, 근육통)을 완화하는 효과가 있습니다.

[주의사항]
- 매일 세잔 이상 정기적 음주자가 복용 시 의사 또는 약사와 상의해야 합니다. 간손상을 일으킬 수 있습니다. 급성전신발진고름물집증, 스티븐스-존슨증후군, 독성표피괴사용해와 같은 중대한 피부반응이 나타날 경우 즉시 복용을 중단해야 합니다. 아세트아미노펜으로 일일 최대 용량(4,000 mg)을 초과하여 복용하지 말아야 합니다. 아세트아미노펜을 포함하는 다른 제품과 함께 복용하지 말아야 합니다.

[추가 안내]
- 매 4~6시간마다 필요시 1포씩 복용하며, 24시간 동안 최대 4포를 초과하지 않도록 합니다. 뜨거운 물에 녹여 복용합니다.운전 및 기계조작 시 주의가 필요합니다.

셀6 표현 다양성 테스트

In [ ]:
# ==============================
# 🧪 표현 다양성 테스트
# ==============================

paraphrase_questions = [
    "소화불량인데 먹을 수 있는 약 있어?",
    "속이 더부룩한데 약 추천해줘",
    "배가 체한 느낌인데 먹을 약 있을까?",
    "소화가 잘 안 되는데 약 뭐 먹어?",
    "과식했는데 속이 불편해"
]

results = []

print("=" * 70)
print("🧪 표현 다양성 테스트 (같은 의미, 다른 질문)")
print("=" * 70)

for i, question in enumerate(paraphrase_questions, 1):
    print("\n" + "-" * 70)
    print(f"[질문 {i}] {question}")
    print("-" * 50)

    response = query_engine.query(question)
    answer = response.response

    print(answer)

    # 추천 약 이름 추출
    drug_name = "추출 실패"
    if "[추천 약]" in answer:
        lines = answer.split("\n")
        for j in range(len(lines)):
            if "[추천 약]" in lines[j]:
                if j + 1 < len(lines):
                    drug_name = lines[j+1].replace("-", "").strip()

    results.append(drug_name)

# 결과 요약
print("\n" + "=" * 70)
print("📊 결과 요약")
print(results)

if len(set(results)) == 1:
    print("✅ 동일한 약 추천 (일관성 있음)")
else:
    print("❌ 표현에 따라 다른 약 추천됨")

출력 결과
======================================================================
🧪 표현 다양성 테스트 (같은 의미, 다른 질문)
======================================================================

----------------------------------------------------------------------
[질문 1] 소화불량인데 먹을 수 있는 약 있어?
--------------------------------------------------
[추천 약]
- 노루모에프내복액

[효능·효과]
- 위산과다, 속쓰림, 위부불쾌감, 위부팽만감, 체함, 구역, 구토, 위통, 신트림, 식욕감퇴(식욕부진), 소화불량, 과식에 사용

[용법·용량]
- 만 15세 이상은 1회 1병 1일 3회, 만 11세 이상~만 15세 미만은 1회 2/3병 1일 3회, 만 8세 이상~만 11세 미만은 1회 1/2병 1일 3회, 만 5세 이상~만 8세 미만은 1회 1/3병 1일 3회, 만 3세 이상~만 5세 미만은 1회 1/4병 1일 3회 식후 복용, 복용간격을 4시간 이상으로 함

[주의사항]
- 만 3개월 미만의 젖먹이는 이 약을 복용하지 마십시오. 이 약을 복용하기 전에 만 1세 미만의 젖먹이, 카라멜에 과민증, 신장장애 환자, 나트륨 제한 식이를 하는 사람은 의사 또는 약사와 상의하십시오. 정해진 용법과 용량을 잘 지키십시오. 어린이에게 투여할 경우 보호자의 지도 감독하에 투여하십시오. 2주 정도 투여하여도 증상의 개선이 없을 경우 복용을 즉각 중지하고 의사 또는 약사와 상의하십시오.

[부작용]
- 변비 또는 설사 등이 나타나는 경우 복용을 즉각 중지하고 의사 또는 약사와 상의하십시오.

[성분]
- 탄산수소나트륨, 글리신, L-멘톨, 고추틴크

[제형(외형)]
- 액제

[추가 안내]
- 2주 정도 복용하여도 증상의 개선이 없을 경우 의사 또는 약사와 상의하십시오.

----------------------------------------------------------------------
[질문 2] 속이 더부룩한데 약 추천해줘
--------------------------------------------------
[추천 약]
- 노루모에프내복액

[효능·효과]
- 위산과다, 속쓰림, 위부불쾌감, 위부팽만감, 체함, 구역, 구토, 위통, 신트림, 식욕감퇴(식욕부진), 소화불량, 과식에 사용

[용법·용량]
- 만 15세 이상은 1회 1병 1일 3회, 만 11세 이상~만 15세 미만은 1회 2/3병 1일 3회, 만 8세 이상~만 11세 미만은 1회 1/2병 1일 3회, 만 5세 이상~만 8세 미만은 1회 1/3병 1일 3회, 만 3세 이상~만 5세 미만은 1회 1/4병 1일 3회 식후 복용

[주의사항]
- 만 3개월 미만의 젖먹이는 복용하지 마십시오. 만 1세 미만의 젖먹이, 카라멜에 과민증, 신장장애 환자, 나트륨 제한 식이를 하는 사람은 의사 또는 약사와 상의하십시오. 어린이에게 투여할 경우 보호자의 지도 감독하에 투여하십시오. 2주 정도 투여하여도 증상의 개선이 없을 경우 복용을 즉각 중지하고 의사 또는 약사와 상의하십시오.

[부작용]
- 변비 또는 설사 등이 나타나는 경우 복용을 즉각 중지하고 의사 또는 약사와 상의하십시오.

[성분]
- 탄산수소나트륨, 글리신, L-멘톨, 고추틴크

[제형(외형)]
- 액제

[추가 안내]
- 속이 더부룩한 증상에 적합한 노루모에프내복액을 추천드립니다. 식후 복용하고 주의사항을 잘 지켜주세요. 만약 부작용이 나타난다면 즉시 의사 또는 약사와 상의하시기 바랍니다.

----------------------------------------------------------------------
[질문 3] 배가 체한 느낌인데 먹을 약 있을까?
--------------------------------------------------
[추천 약]
- 훼스탈골드정

[효능·효과]
- 소화불량, 식욕감퇴(식욕부진), 과식, 체함, 소화촉진, 소화불량으로 인한 위부팽만감에 사용

[용법·용량]
- 성인은 1회 1정씩, 1일 3회 식후에 씹지말고 복용

[주의사항]
- 만 7세 이하의 어린이는 복용하지 말 것
- 알레르기 체질인 환자, 임부 또는 임신 가능성이 있는 여성은 의사 또는 약사와 상의
- 정해진 용법과 용량을 잘 지킬 것
- 2주 정도 복용하여도 증상의 개선이 없을 경우 의사 또는 약사와 상의

[부작용]
- (부작용 정보 없음)

[성분]
- 판크레아틴, 디아스타제·프로테아제·셀룰라제, 프로자임6, 리파제, 셀룰라제, 우르소데옥시콜산, 시메티콘

[제형(외형)]
- 장용성필름코팅정, 장방형, 하양

[추가 안내]
- 만약 2주 정도 복용하여도 증상이 개선되지 않는다면 의사 또는 약사와 상의하시기 바랍니다.

----------------------------------------------------------------------
[질문 4] 소화가 잘 안 되는데 약 뭐 먹어?
--------------------------------------------------
[추천 약]
- 훼스탈슈퍼자임정

[효능·효과]
- 소화불량, 식욕감퇴, 과식, 체함, 소화촉진 등에 사용

[용법·용량]
- 만 15세 이상 및 성인은 1회 1~2정씩, 1일 3회 복용

[주의사항]
- 만 7세 이하 어린이는 복용하지 말 것
- 신장장애 환자, 알레르기 체질 등은 의사와 상의할 것
- 어린이에게 투여 시 보호자 지도 감독 필요

[부작용]
- 변비 또는 설사 등이 나타날 경우 의사와 상의할 것

[성분]
- 리파제, 프로자임, 디아스타제·프로테아제·셀룰라제, 시메티콘, 산화마그네슘, 수용성아줄렌, 당약 가루

[제형(외형)]
- 필름코팅정, 장방형, 하양

[추가 안내]
- 소화불량 증상이 지속될 경우 의사와 상의하시기 바랍니다.

----------------------------------------------------------------------
[질문 5] 과식했는데 속이 불편해
--------------------------------------------------
[추천 약]
- 노루모에프내복액

[효능·효과]
- 소화불량, 속쓰림, 위부불쾌감, 위부팽만감, 체함, 구역, 구토, 위통, 신트림, 식욕감퇴(식욕부진), 과식에 사용

[용법·용량]
- 만 15세 이상은 1회 1병 1일 3회, 만 11세 이상~만 15세 미만은 1회 2/3병 1일 3회, 만 8세 이상~만 11세 미만은 1회 1/2병 1일 3회, 만 5세 이상~만 8세 미만은 1회 1/3병 1일 3회, 만 3세 이상~만 5세 미만은 1회 1/4병 1일 3회 식후 복용

[주의사항]
- 만 3개월 미만의 젖먹이는 복용하지 마세요. 복용 전에 의사 또는 약사와 상의하세요. 정해진 용법과 용량을 잘 지켜야 합니다.

[부작용]
- 변비 또는 설사 등이 나타날 경우 복용을 중지하고 의사 또는 약사와 상의하세요.

[성분]
- 탄산수소나트륨, 글리신, L-멘톨, 고추틴크

[제형(외형)]
- 액제

[추가 안내]
- 속이 불편하신 경우 위장관을 편안하게 해주는 노루모에프내복액을 드시면 도움이 될 수 있습니다. 만약 증상이 계속되거나 심해진다면 의사와 상의해보세요.

======================================================================
📊 결과 요약
['노루모에프내복액', '노루모에프내복액', '훼스탈골드정', '훼스탈슈퍼자임정', '노루모에프내복액']
❌ 표현에 따라 다른 약 추천됨
